# 4DMAG pump-probe analysis

What this notebook does, on any lab's data server (nothing here is tied to one run or one lab):

1. load a pump-probe delay scan (newest one, or by scan number / uid prefix)
2. pump-on / pump-off signal per delay, mean ± standard error over repeats (`plot_data` with `group` + `average`)
3. difference, ratio and relative difference pumped vs unpumped (`combine`)
4. main / reference camera ROI ratios, recomputed with a **background (BG) run** subtracted
5. camera frames of one point (`run.frames`)
6. export to CSV / HDF5
7. several runs compared (overlay of their difference curves)
8. exponential decay fitted to the difference, fit drawn into the same plot and saved

Settings: `FASTQ_LAB` (lab file; default `control/lab.yaml`) selects the data server. Runs are found by
`run_kind` (`TimeDelay` for `fdm_delay_scan`, `BG` for `fdm_background`) and by the columns they need, so
an `nd_scan` built in the scan builder works too as long as it has a pump group and ROI ratios.

In [ ]:
%matplotlib inline
from nbsetup import *
from fastq.analysis import plot_data
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})

## 1. The run

`RUN = None` takes the newest run of `KIND` that has a pump group and main/ref ratio columns.
Otherwise a scan number (`905`), `-1` (latest run of any kind) or a uid prefix (`"1e0ba67f"`).

In [ ]:
RUN = None
KIND = "TimeDelay"        # run_kind of fdm_delay_scan
GROUP = "pump_shutter"    # pump on / off column (the pump axis of the scan)

run = pick_run(RUN, kind=KIND, columns=[GROUP, "*_over_*"])
hints = run.md.get("plot_hints") or {}
X = hints.get("x") or "delay"
GROUP = hints.get("group") or GROUP
roles = run.md.get("roles") or {}
dets = run.md.get("detectors") or []
MAIN = roles.get("main_camera") or dets[0]
REF = roles.get("ref_camera") or dets[1]
df = run.table()
RATIOS = [c for c in df.columns if c.startswith(f"{MAIN}_over_{REF}_")]
ROIS = [c[len(f"{MAIN}_over_{REF}_"):] for c in RATIOS]
# in the camera's order (its configuration recorded in the run), else as found
conf = (run.node["primary"].metadata or {}).get("configuration") or {}
order = ((conf.get(MAIN) or {}).get("data") or {}).get(f"{MAIN}_roi_names")
if order:
    import json
    ROIS = [r for r in json.loads(order) if r in ROIS] + [r for r in ROIS if r not in json.loads(order)]
    RATIOS = [f"{MAIN}_over_{REF}_{r}" for r in ROIS]
# the plan's own choice of signal (what the dashboard plots first), else the first ratio
SIGNAL = next((p["y"][0] for p in default_plots(run.md, {"primary": data_keys(run.node)}) if p.get("y")), RATIOS[0])

print(run)
print(f"sample {run.md.get('sample')!r}, {len(df)} points, x = {X}, group = {GROUP} {sorted(df[GROUP].unique())}")
print(f"cameras {MAIN} / {REF}, ROIs {ROIS}")
print(f"signal {SIGNAL}")
print("plan call:", run.md.get("plan_request"))

## 2. Pump on / pump off per delay

`plot_data(df, spec)` is the same function the dashboard uses: `group` makes one curve per pump state,
`average` collapses the repeats of each delay into mean ± s.e.m. (standard error = std / √n).

In [ ]:
spec = {"x": X, "y": [SIGNAL], "group": GROUP, "average": True, "title": f"{label(run)}: pump on / off"}
onoff = plot_data(df, spec)
fig, ax = plt.subplots()
draw(ax, onoff)
plt.show()

# the same numbers as a table
pd.DataFrame({s["name"]: pd.Series(s["y"], index=s["x"]) for s in onoff["series"]}).rename_axis(X)

## 3. Difference and ratio

`combine` with a two-valued group: `diff` = on − off, `ratio` = on / off, `rel_diff` = (on − off) / off,
errors propagated from the s.e.m. of both curves.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
combined = {}
for ax, how in zip(axes, ("diff", "ratio", "rel_diff")):
    combined[how] = plot_data(df, {"x": X, "y": [SIGNAL], "group": GROUP, "combine": how, "title": how})
    draw(ax, combined[how])
    ax.legend().remove()
fig.suptitle(f"{label(run)} - {SIGNAL}")
fig.tight_layout()
plt.show()

## 4. Main / reference ROI ratios and background subtraction

Each camera reports one value per ROI (`<cam>_<roi>`: mean over the ROI minus the camera offset minus the
background slice, configured by `fdm_set_rois`); the run also carries `<main>_over_<ref>_<roi>` = main / ref
per point. Check that:

In [ ]:
roi = SIGNAL[len(f"{MAIN}_over_{REF}_"):] if SIGNAL.startswith(f"{MAIN}_over_{REF}_") else ROIS[0]
recomputed = df[f"{MAIN}_{roi}"] / df[f"{REF}_{roi}"]
print(f"ROI {roi}: max |recorded - main/ref| =", float(np.nanmax(np.abs(recomputed - df[f'{MAIN}_over_{REF}_{roi}']))))

# all ROIs at once: relative pump effect (on - off) / off per ROI and delay
rel = {r: plot_data(df, {"x": X, "y": [f"{MAIN}_over_{REF}_{r}"], "group": GROUP, "combine": "rel_diff"})
       for r in ROIS}
rel = {r: o["series"][0] for r, o in rel.items() if o["series"]}
mat = np.array([s["y"] for s in rel.values()], dtype=float)
fig, ax = plt.subplots(figsize=(8, 0.25 * len(rel) + 1.5))
m = ax.pcolormesh(rel[next(iter(rel))]["x"], np.arange(len(rel)), mat, shading="nearest", cmap="RdBu_r",
                  vmin=-np.nanmax(np.abs(mat)), vmax=np.nanmax(np.abs(mat)))
ax.set_yticks(np.arange(len(rel)), list(rel))
ax.set_xlabel(X); ax.set_ylabel("ROI"); plt.colorbar(m, label="(on - off) / off")
ax.set_title(f"{label(run)}: relative pump effect per ROI")
plt.show()

**Background run.** A `BG` run (`fdm_background`) records the cameras without the signal of interest.
Its mean per ROI is subtracted from each camera before the ratio is formed:
`(main − BG_main) / (ref − BG_ref)`. Taken: the newest BG run recorded before this run (same sample if there
is one), or set `BG_RUN`.

In [ ]:
BG_RUN = None
need = [f"{MAIN}_{roi}", f"{REF}_{roi}"]
if BG_RUN is not None:
    bg = pick_run(BG_RUN, required=False, columns=need)
else:
    cands = [r for r in find_runs(kind="BG", columns=need, limit=50) if r.md.get("time", 0) <= run.md.get("time", 0)]
    same = [r for r in cands if r.md.get("sample") == run.md.get("sample")]
    bg = (same or cands or [None])[0]
    if bg is None:
        print(f"(skipped) no BG run with columns {need} before {run} - record one with fdm_background")

if bg is not None:
    bgt = bg.table()
    print("background:", bg, f"{len(bgt)} points")
    level = pd.DataFrame({"main": [bgt[f"{MAIN}_{r}"].mean() for r in ROIS],
                          "ref": [bgt[f"{REF}_{r}"].mean() for r in ROIS]}, index=ROIS)
    display(level.T.round(3))
    for r in ROIS:
        df[f"bgsub_{r}"] = (df[f"{MAIN}_{r}"] - level.at[r, "main"]) / (df[f"{REF}_{r}"] - level.at[r, "ref"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    draw(axes[0], plot_data(df, {"x": X, "y": [SIGNAL, f"bgsub_{roi}"], "group": GROUP, "average": True,
                                 "title": "pump on / off, raw vs BG-subtracted"}))
    draw(axes[1], plot_data(df, {"x": X, "y": [SIGNAL, f"bgsub_{roi}"], "group": GROUP, "combine": "diff",
                                 "title": "difference"}))
    fig.tight_layout()
    plt.show()

## 5. Camera frames of one point

`run.frames("<cam>_image")` is the frame array of the run (one frame per point, from the camera's HDF5 file,
served by Tiled); index it to read only what is needed. The ROI ranges come from the camera configuration
recorded in the run.

In [ ]:
POINT = None   # row of df; None = the first pump-on point at the delay with the largest |difference|
import json

key = f"{MAIN}_image"
d = combined["diff"]["series"][0]
x_peak = d["x"][int(np.nanargmax(np.abs(np.array(d["y"], dtype=float))))]
on_level = max(df[GROUP].unique())
if POINT is None:
    POINT = int(np.flatnonzero((df[X] == x_peak) & (df[GROUP] == on_level))[0])
try:
    frames = {cam: run.frames(f"{cam}_image") for cam in (MAIN, REF)}
    conf = (run.node["primary"].metadata or {}).get("configuration") or {}
    fig, axes = plt.subplots(2, 2, figsize=(14, 5), gridspec_kw={"height_ratios": [1, 1.4]})
    for col, (cam, arr) in enumerate(frames.items()):
        img = np.squeeze(np.asarray(arr[POINT], dtype=float))
        axes[0, col].imshow(np.atleast_2d(img), aspect="auto", cmap="viridis")
        axes[0, col].set_title(f"{cam}, point {POINT} ({X}={df[X].iloc[POINT]:g}, {GROUP}={df[GROUP].iloc[POINT]:g})")
        spec_ = np.atleast_2d(img).mean(axis=0)
        axes[1, col].plot(spec_, lw=0.8)
        rois = json.loads(((conf.get(cam) or {}).get("data") or {}).get(f"{cam}_rois") or "{}")
        for r in rois.get("rois", []):
            if r["name"] in (roi,):
                axes[1, col].axvspan(*r["x"], color="C1", alpha=0.25, label=f"ROI {r['name']}")
        if rois.get("background"):
            axes[1, col].axvspan(*rois["background"], color="0.5", alpha=0.25, label="background slice")
        axes[1, col].set_xlabel("pixel"); axes[1, col].set_ylabel("counts (row mean)"); axes[1, col].legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    print("frame array:", frames[MAIN], frames[MAIN].shape)
except Exception as exc:
    print(f"(skipped) frames not readable: {type(exc).__name__}: {exc}\n"
          "the camera's HDF5 folder must be in data.readable_storage of the lab file")

In [ ]:
# pump-on minus pump-off spectrum at that delay (frames averaged over repeats)
try:
    sel = lambda lv: np.flatnonzero((df[X] == x_peak) & (df[GROUP] == lv))
    levels = sorted(df[GROUP].unique())
    mean_frame = lambda idx: np.mean([np.atleast_2d(np.squeeze(np.asarray(frames[MAIN][int(i)], dtype=float))).mean(axis=0)
                                      for i in idx], axis=0)
    off, on = mean_frame(sel(levels[0])), mean_frame(sel(levels[-1]))
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(on - off, lw=0.8)
    ax.set_title(f"{MAIN}: pump on - off at {X} = {x_peak:g} ({len(sel(levels[-1]))} + {len(sel(levels[0]))} frames)")
    ax.set_xlabel("pixel"); ax.set_ylabel("Δ counts")
    plt.show()
except NameError:
    print("(skipped) no frames")

## 6. Export

- the table of the run (all scalar columns, derived values included) as CSV,
- the averaged curves as CSV,
- the whole run as HDF5 (`export_hdf5`: start / stop metadata as JSON attributes, one group per stream;
  `frames=True` copies the camera frames too, otherwise their source is noted as an attribute).

Files go to `notebooks/exports/` (git-ignored).

In [ ]:
import h5py

EXPORTS.mkdir(exist_ok=True)
stem = EXPORTS / f"run{run.md.get('scan_id')}_{run.uid[:8]}"
df.to_csv(f"{stem}.csv", index=False)
curves = pd.DataFrame({s["name"]: pd.Series(s["y"], index=s["x"]) for s in onoff["series"]})
for s in combined["diff"]["series"]:
    curves[s["name"]] = pd.Series(s["y"], index=s["x"])
    curves[s["name"] + " err"] = pd.Series(s["err"], index=s["x"])
curves.rename_axis(X).to_csv(f"{stem}_curves.csv")
h5 = export_hdf5(run, f"{stem}.h5", frames=False)
with h5py.File(h5) as f:
    print(h5.name, "groups:", list(f), "| primary:", len(f["primary"]), "datasets,",
          len(f["primary"][SIGNAL]), "points | frames:", f["primary"].attrs.get(f"{MAIN}_image.source"))
print(f"{stem.name}.csv, {stem.name}_curves.csv written to {EXPORTS}")

## 7. Several runs compared

The newest `N_COMPARE` runs of the same kind with the same signal and group (optionally of the same sample),
their difference curves overlaid.

In [ ]:
N_COMPARE = 4
SAME_SAMPLE = False
others = find_runs(kind=run.md.get("run_kind"), columns=[SIGNAL, GROUP], limit=N_COMPARE,
                   sample=run.md.get("sample") if SAME_SAMPLE else None)
if run.uid not in [r.uid for r in others]:
    others = [run] + others[: N_COMPARE - 1]
fig, ax = plt.subplots()
for r in others:
    o = r.plot({"x": X, "y": [SIGNAL], "group": GROUP, "combine": "diff"})
    for s in o["series"]:
        ax.errorbar(s["x"], s["y"], yerr=s["err"], marker="o", ms=3, capsize=2, label=label(r))
ax.axhline(0, color="0.6", lw=0.5)
ax.set_xlabel(X); ax.set_ylabel(f"Δ {SIGNAL} (on - off)"); ax.legend(fontsize=8)
ax.set_title(f"{len(others)} runs of kind {run.md.get('run_kind')!r}")
plt.show()

## 8. Exponential decay fit of the difference

Model for delays ≥ t₀: Δ(t) = A · exp(−(t − t₀)/τ) + C, weighted by the s.e.m. of each point
(`scipy.optimize.curve_fit`). `T0` = None takes the delay of the largest |Δ|. The fit is added as one more
series to the plot data (the same structure `plot_data` returns), drawn, and saved as PNG + CSV.

On the simulator there is no real pump effect - the fit then just describes noise (large errors).

In [ ]:
from scipy.optimize import curve_fit

T0 = None
d = combined["diff"]["series"][0]
t, y, e = (np.array(d[k], dtype=float) for k in ("x", "y", "err"))
t0 = t[int(np.nanargmax(np.abs(y)))] if T0 is None else T0
m = (t >= t0) & np.isfinite(y)
model = lambda tt, a, tau, c: a * np.exp(-(tt - t0) / tau) + c
fit_out = dict(combined["diff"], title=f"{label(run)}: Δ {SIGNAL} with exponential fit")
try:
    if m.sum() < 4:
        raise RuntimeError(f"only {m.sum()} points at delays >= {t0:g}")
    sigma = e[m] if np.all(e[m] > 0) else None
    span = max(t[m].max() - t0, 1e-9)
    p, cov = curve_fit(model, t[m], y[m], p0=[y[m][0] - y[m][-1], span / 3, y[m][-1]], sigma=sigma,
                       absolute_sigma=sigma is not None, bounds=([-np.inf, span / 1000, -np.inf], [np.inf, 100 * span, np.inf]),
                       maxfev=20000)
    perr = np.sqrt(np.diag(cov))
    fit = {"A": p[0], "A_err": perr[0], "tau": p[1], "tau_err": perr[1], "C": p[2], "C_err": perr[2], "t0": t0,
           "chi2_red": float(np.sum(((y[m] - model(t[m], *p)) / (sigma if sigma is not None else 1)) ** 2) / max(m.sum() - 3, 1))}
    tt = np.linspace(t0, t[m].max(), 200)
    fit_out["series"] = list(combined["diff"]["series"]) + [
        {"name": f"fit: τ = {p[1]:.3g} ± {perr[1]:.2g} {X}, A = {p[0]:.3g} ± {perr[0]:.2g}", "x": tt.tolist(),
         "y": model(tt, *p).tolist()}]
    print(pd.Series(fit).to_string())
except (RuntimeError, ValueError) as exc:
    fit = None
    print(f"fit failed: {exc}")
fig, ax = plt.subplots()
draw(ax, fit_out)
ax.axvline(t0, color="0.6", lw=0.5, ls="--")
plt.show()
if fit:
    fig.savefig(f"{stem}_fit.png", dpi=120)
    pd.Series(fit).to_csv(f"{stem}_fit.csv", header=["value"])
    print(f"saved {stem.name}_fit.png / _fit.csv")